# BiRefNet Studio • Colab Forms • L4 GPU • UV-only

Based on your uploaded notebook and [orioninsist/BiRefNet](https://github.com/orioninsist/BiRefNet). **Run in Google Colab browser** for Colab `# @param` drop-downs (View → Show/hide code). When connected through **VS Code**, these form annotations may display as Python source; the runtime can still execute the notebook.

**Recommended:** Runtime → Change runtime type → L4 GPU (if available). High-RAM is optional, not required for basic inference. The notebook detects available GPU and CUDA; it cannot reserve L4. No `pip` executable is used, only UV. Google Drive holds model files, images, and results across sessions. The isolated environment must be recreated on each new Colab VM.

**Workflow:** 1. Mount Drive → 2. Choose storage/settings → 3. Check GPU → 4. UV setup → 5. Put images in Drive → 6. Choose model/settings → 7. Process → 8. Review results.

## 1 · Mount Google Drive
Run this once per Colab runtime. In VS Code + remote Colab, authorize Drive through the Colab browser if the mount cannot open its consent flow.

In [ ]:
# @title 1 • Connect Google Drive
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive', force_remount=False)
assert Path('/content/drive/MyDrive').exists(), 'Drive mount is unavailable. Authorize Drive in the Colab browser.'
print('Google Drive is connected')


## 2 · Storage and dependency preferences
Change the text input only if you want a different folder. Avoid moving cache paths after downloading models. The default uses local SSD for active processing, while Drive stores reusable weights and final results.

In [ ]:
# @title 2 • Storage & runtime preferences
project_folder = 'BiRefNet_Colab' # @param {type:'string'}
keep_uploaded_inputs_in_drive = True # @param {type:'boolean'}
use_local_model_copy = True # @param {type:'boolean'}
strict_l4_only = False # @param {type:'boolean'}
install_dependencies = True # @param {type:'boolean'}

from pathlib import Path
import json, re
assert re.fullmatch(r'[a-zA-Z0-9_ -]{1,64}', project_folder), 'Use a simple Drive folder name (letters/numbers/space/_/-).'
ROOT = Path('/content/drive/MyDrive') / project_folder
for subfolder in ('inputs','outputs','models','hf_cache'):
    (ROOT / subfolder).mkdir(parents=True, exist_ok=True)
CFG_PATH=Path('/content/birefnet_config.json')
CFG={'drive_root':str(ROOT),'local_input':'/content/birefnet_input','local_model':'/content/birefnet_model','venv':'/content/birefnet-venv','keep_uploaded_inputs_in_drive':keep_uploaded_inputs_in_drive,'use_local_model_copy':use_local_model_copy,'strict_l4_only':strict_l4_only}
CFG_PATH.write_text(json.dumps(CFG,indent=2))
Path(CFG['local_input']).mkdir(parents=True,exist_ok=True)
print('Persistent Drive folder:', ROOT)
print('Upload images to:', ROOT / 'inputs')


## 3 · Verify CUDA and GPU
**L4 ~24 GB VRAM** is a good choice; other GPUs can work, unless you turn on *strict L4 only*. Driver and CUDA support must also match the installed PyTorch build.

In [ ]:
# @title 3 • Check GPU
import subprocess, json
from pathlib import Path
cfg=json.loads(Path('/content/birefnet_config.json').read_text())
check=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True,capture_output=True)
print(check.stdout or check.stderr)
assert check.returncode==0, 'No NVIDIA GPU attached. Choose a GPU runtime.'
if cfg['strict_l4_only']:
    assert 'L4' in check.stdout, 'Strict L4 option enabled but GPU is not L4.'
print('GPU check passed')


## 4 · UV environment setup (no pip executable)
Installs **Python 3.11**, a private virtual environment, and matched CUDA PyTorch wheels via **UV**. The original notebook used fixed CUDA 12.4 wheels; that may fail with newer runtimes. Select a CUDA wheel build supported by the actual driver. Dependencies are installed locally on every fresh VM, model weights remain on Drive. All model-processing commands later use the UV interpreter directly.

**If PyTorch installation fails**, review the installation output for driver/wheel compatibility. Do not rely on Colab's system `python` to run the model.

In [ ]:
# @title 4 • Install UV / CUDA PyTorch
# @markdown UV installer is used; no pip command.
cuda_wheels = 'cu124' # @param ['cu124','cu126']
run_install = True # @param {type:'boolean'}

import os, subprocess, shutil
from pathlib import Path
venv=Path('/content/birefnet-venv')
env=os.environ.copy()
env['PATH']=str(Path.home()/'.local/bin')+os.pathsep+env.get('PATH','')
env['UV_CACHE_DIR']='/content/uv_cache'
def run(*args):
    print('+', ' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),check=True,env=env)
if run_install:
    if not shutil.which('uv',path=env['PATH']):
        run('bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh')
    uv=shutil.which('uv',path=env['PATH'])
    assert uv, 'UV installation did not put uv on PATH.'
    run(uv,'--version')
    run(uv,'python','install','3.11')
    if not (venv/'bin/python').is_file():
        run(uv,'venv','--python','3.11',str(venv))
    py=str(venv/'bin/python')
    run(uv,'pip','install','--python',py,'--index-url',f'https://download.pytorch.org/whl/{cuda_wheels}','torch==2.6.0','torchvision==0.21.0')
    run(uv,'pip','install','--python',py,'transformers==4.49.0','huggingface-hub>=0.29,<1','timm>=1.0,<2','safetensors>=0.4,<1','accelerate>=1,<2','einops>=0.7,<1','kornia>=0.7,<1','Pillow>=10,<12','numpy>=1.26,<2','opencv-python-headless>=4.8,<5','scipy>=1.11,<2','scikit-image>=0.22,<1')
else:
    py=str(venv/'bin/python')
assert Path(py).is_file(), 'Run installation at least once for each new Colab runtime.'
run(py,'-c','import torch,torchvision,transformers,timm; print("Torch",torch.__version__,"Torchvision",torchvision.__version__,"Transformers",transformers.__version__); assert torch.cuda.is_available(), "CUDA unavailable"')


## 5 · Choose a BiRefNet model
Pick based on your subject. **General** is the safest starting point; **Lite** is quicker, **Matting / HR-matting** suit alpha edges, **HR** targets larger images, and **Dynamic** accepts a wide range of scales. Different model repos have different memory requirements. Each chosen model is cached separately on Drive and reused; the first download may be large. All listed entries are official [ZhengPeng7 models](https://huggingface.co/ZhengPeng7).

In [ ]:
# @title 5 • Model & download choices
model_choice = 'General — balanced (1024)' # @param ['General — balanced (1024)','Lite — fastest (1024)','Dynamic — flexible resolution','HR — high resolution (2048)','Matting — soft edges (1024)','HR-matting — fine edges (2048)']
force_redownload = False # @param {type:'boolean'}
copy_model_to_local_ssd = True # @param {type:'boolean'}

import json, os, subprocess, shutil, tempfile
from pathlib import Path
cfg=json.loads(Path('/content/birefnet_config.json').read_text())
models={
 'General — balanced (1024)':('ZhengPeng7/BiRefNet',1024),
 'Lite — fastest (1024)':('ZhengPeng7/BiRefNet_lite',1024),
 'Dynamic — flexible resolution':('ZhengPeng7/BiRefNet_dynamic',1024),
 'HR — high resolution (2048)':('ZhengPeng7/BiRefNet_HR',2048),
 'Matting — soft edges (1024)':('ZhengPeng7/BiRefNet-matting',1024),
 'HR-matting — fine edges (2048)':('ZhengPeng7/BiRefNet_HR-matting',2048),
}
repo,default_resolution=models[model_choice]
root=Path(cfg['drive_root'])
folder=root/'models'/repo.replace('/','__')
folder.mkdir(parents=True,exist_ok=True)
py=cfg['venv']+'/bin/python'
assert Path(py).exists(), 'First run UV installation cell.'
program="""from huggingface_hub import snapshot_download
from pathlib import Path
import os,sys
folder=Path(sys.argv[1]); repo=sys.argv[2]; refresh=sys.argv[3]=='yes'
ready=folder/'.ready_commit'
weights=list(folder.glob('*.safetensors'))+list(folder.glob('*.bin'))
if not ready.exists() or not weights or refresh:
    print('Downloading / verifying:',repo,flush=True)
    p=snapshot_download(repo_id=repo,local_dir=str(folder),allow_patterns=['*.json','*.py','*.safetensors','*.bin','*.txt','*.md'],max_workers=4)
    weights=list(folder.glob('*.safetensors'))+list(folder.glob('*.bin'))
    assert weights,'No model weights found in downloaded repository.'
    from huggingface_hub import HfApi
    ready.write_text(HfApi().model_info(repo).sha)
else:
    print('Using cached model:',ready.read_text().strip())
print('Model files:',len(weights))
"""
process=subprocess.run([py,'-c',program,str(folder),repo,'yes' if force_redownload else 'no'],text=True)
process.check_returncode()
local=Path('/content/birefnet_model')
if copy_model_to_local_ssd:
    if local.exists(): shutil.rmtree(local)
    shutil.copytree(folder,local,symlinks=False)
    load_path=local
else: load_path=folder
selection={'repo':repo,'model_choice':model_choice,'default_resolution':default_resolution,'model_path':str(load_path),'revision':(folder/'.ready_commit').read_text().strip()}
Path('/content/birefnet_model_selection.json').write_text(json.dumps(selection,indent=2))
print('Selected:',repo,'at',load_path)


## 6 · Input location (recommended: Google Drive for VS Code)
The Colab browser `files.upload()` widget does **not** reliably appear in VS Code connected to Colab. Default: upload your images via the Drive web UI to `MyDrive/BiRefNet_Colab/inputs` and select **Google Drive folder**. You may alternatively use Colab's browser upload chooser when running the notebook **inside Colab browser**.

In [ ]:
# @title 6 • Input images
image_source = 'Google Drive folder (works with VS Code)' # @param ['Google Drive folder (works with VS Code)','Colab browser upload','Existing runtime folder']
clear_previous_runtime_inputs = True # @param {type:'boolean'}
recursive_drive_search = False # @param {type:'boolean'}

from pathlib import Path
import shutil, json
cfg=json.loads(Path('/content/birefnet_config.json').read_text())
inp=Path(cfg['local_input']); inp.mkdir(parents=True,exist_ok=True)
allowed={'.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff'}
if clear_previous_runtime_inputs:
    for old in inp.iterdir():
        if old.is_file() and old.suffix.lower() in allowed: old.unlink()
if image_source.startswith('Google Drive'):
    src=Path(cfg['drive_root'])/'inputs'
    src.mkdir(parents=True,exist_ok=True)
    images=src.rglob('*') if recursive_drive_search else src.iterdir()
    for image in images:
        if image.is_file() and image.suffix.lower() in allowed:
            # Preserve relative folder names safely in the single runtime directory.
            name=image.name if not recursive_drive_search else '__'.join(image.relative_to(src).parts)
            shutil.copy2(image,inp/name)
elif image_source=='Colab browser upload':
    from google.colab import files
    uploaded=files.upload()
    for name,data in uploaded.items():
        safe=Path(name).name
        if Path(safe).suffix.lower() in allowed:
            (inp/safe).write_bytes(data)
            if cfg['keep_uploaded_inputs_in_drive']:
                (Path(cfg['drive_root'])/'inputs'/safe).write_bytes(data)
else:
    print('Using images already in',inp)
images=[p for p in inp.iterdir() if p.is_file() and p.suffix.lower() in allowed]
print('Images ready:',len(images),'in',inp)
if not images: print('Add images to',Path(cfg['drive_root'])/'inputs','then rerun this cell.')


## 7 · Processing settings (all controls use `# @param`)
**Resolution:** 1024 is reliable for General/Lite/Matting; 2048 matches HR/HR-matting but may require significantly more VRAM. **FP16** is recommended on L4. **Soft** alpha is best for hair/fine edges; **Binary** removes transparency gradients. Output masks are resized to original image dimensions. Matte edge quality depends on the chosen pretrained model and input quality, not just these sliders.

In [ ]:
# @title 7 • Image quality, GPU & output options
input_resolution = 'Model recommended' # @param ['Model recommended','512','768','1024','1280','1536','2048']
precision = 'FP16 — recommended' # @param ['FP16 — recommended','FP32 — full precision','BF16 — if supported']
mask_style = 'Soft alpha — natural edges' # @param ['Soft alpha — natural edges','Binary alpha — sharp edges']
binary_threshold = 0.5 # @param {type:'slider', min:0.05, max:0.95, step:0.05}
mask_blur_radius = 0.0 # @param {type:'slider', min:0.0, max:5.0, step:0.5}
mask_contrast = 1.0 # @param {type:'slider', min:0.5, max:2.5, step:0.1}
save_transparent_png = True # @param {type:'boolean'}
save_grayscale_mask = True # @param {type:'boolean'}
save_white_background = False # @param {type:'boolean'}
save_comparison_preview = False # @param {type:'boolean'}
automatically_zip_results = True # @param {type:'boolean'}
continue_on_image_error = True # @param {type:'boolean'}

from pathlib import Path
import json
select=json.loads(Path('/content/birefnet_model_selection.json').read_text())
cfg=json.loads(Path('/content/birefnet_config.json').read_text())
resolution=select['default_resolution'] if input_resolution=='Model recommended' else int(input_resolution)
assert any([save_transparent_png, save_grayscale_mask, save_white_background, save_comparison_preview]), 'Enable at least one output type.'
settings=dict(resolution=resolution,precision=precision.split(' — ')[0],mask_style=mask_style.split(' — ')[0],binary_threshold=binary_threshold,mask_blur_radius=mask_blur_radius,mask_contrast=mask_contrast,save_transparent_png=save_transparent_png,save_grayscale_mask=save_grayscale_mask,save_white_background=save_white_background,save_comparison_preview=save_comparison_preview,automatically_zip_results=automatically_zip_results,continue_on_image_error=continue_on_image_error)
Path('/content/birefnet_processing_settings.json').write_text(json.dumps(settings,indent=2))
print('Model:',select['model_choice'],'Resolution:',resolution,'Precision:',settings['precision'],'Mask:',settings['mask_style'])


## 8 · Run background removal
One model load per batch, CUDA inference on the GPU, and each result is saved in a timestamped Google Drive folder. Local Colab SSD is used to process images; your model cache remains on Drive. Use the preview/ZIP options above to control extra outputs. This runs remote Hugging Face model code (`trust_remote_code=True`), only from the explicit official repository you selected.

In [ ]:
# @title 8 • Run BiRefNet
start_processing = True # @param {type:'boolean'}

import subprocess, json
from pathlib import Path
assert start_processing, 'Enable processing and rerun this cell.'
PY=Path('/content/birefnet-venv/bin/python')
assert PY.is_file(), 'UV environment not found.'
script=r"""from pathlib import Path
from datetime import datetime, timezone
import json, sys, traceback, zipfile
from contextlib import nullcontext
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

cfg=json.loads(Path('/content/birefnet_config.json').read_text())
sel=json.loads(Path('/content/birefnet_model_selection.json').read_text())
opts=json.loads(Path('/content/birefnet_processing_settings.json').read_text())
folder=Path(cfg['local_input'])
allowed={'.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff'}
images=sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in allowed)
assert images, 'No images. Upload images to Drive inputs folder and rerun input step.'
assert torch.cuda.is_available(),'GPU not enabled in UV Python'
if opts['precision']=='BF16': assert torch.cuda.is_bf16_supported(),'This GPU cannot use BF16; choose FP16.'
if opts['precision']=='FP16': dtype=torch.float16
elif opts['precision']=='BF16': dtype=torch.bfloat16
else: dtype=None
try: torch.set_float32_matmul_precision('high')
except Exception: pass
print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Loading',sel['repo'],flush=True)
net=AutoModelForImageSegmentation.from_pretrained(sel['model_path'],trust_remote_code=True,local_files_only=True).to('cuda').eval()
size=opts['resolution']
pre=transforms.Compose([transforms.Resize((size,size)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
root=Path(cfg['drive_root'])/'outputs'
out=root/(datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')+'_'+sel['repo'].split('/')[-1])
out.mkdir(parents=True,exist_ok=True)
(out/'settings.json').write_text(json.dumps({'model':sel,'options':opts},indent=2))
success=0
for idx,p in enumerate(images,1):
    try:
        with Image.open(p) as raw: image=ImageOps.exif_transpose(raw).convert('RGB')
        W,H=image.size
        x=pre(image).unsqueeze(0).to('cuda')
        ctx=torch.autocast('cuda',dtype=dtype) if dtype else nullcontext()
        with torch.inference_mode(),ctx:
            pred=net(x)
            logits=pred[-1] if isinstance(pred,(list,tuple)) else pred
            if isinstance(logits,(list,tuple)): logits=logits[-1]
            if isinstance(logits,dict):
                raise TypeError('Unexpected dict output; selected model may require a different inference adapter.')
            mask=logits.sigmoid().float()
            mask=F.interpolate(mask,size=(H,W),mode='bilinear',align_corners=False)
            arr=(mask.squeeze().clamp(0,1).cpu().numpy()*255).astype(np.uint8)
        alpha=Image.fromarray(arr,mode='L')
        if opts['mask_contrast']!=1: alpha=ImageEnhance.Contrast(alpha).enhance(opts['mask_contrast'])
        if opts['mask_blur_radius']>0: alpha=alpha.filter(ImageFilter.GaussianBlur(opts['mask_blur_radius']))
        if opts['mask_style']=='Binary': alpha=alpha.point(lambda v: 255 if v/255>=opts['binary_threshold'] else 0)
        stem=p.stem
        if opts['save_grayscale_mask']: alpha.save(out/f'{stem}_mask.png')
        rgba=image.copy().convert('RGBA'); rgba.putalpha(alpha)
        if opts['save_transparent_png']: rgba.save(out/f'{stem}_transparent.png')
        if opts['save_white_background']:
            bg=Image.new('RGBA',rgba.size,(255,255,255,255)); bg.alpha_composite(rgba); bg.convert('RGB').save(out/f'{stem}_white.png')
        if opts['save_comparison_preview']:
            before=image.copy(); before.thumbnail((700,700))
            after=rgba.copy(); after.thumbnail((700,700))
            canvas=Image.new('RGB',(before.width+after.width,max(before.height,after.height)),(235,235,235))
            canvas.paste(before,(0,0));canvas.paste(after,(before.width,0),after)
            canvas.save(out/f'{stem}_preview.jpg',quality=90)
        success+=1
        print(f'[{idx}/{len(images)}] ✓ {p.name}',flush=True)
    except Exception as error:
        print(f'[{idx}/{len(images)}] FAILED {p.name}: {error}',flush=True)
        if not opts['continue_on_image_error']: raise
    finally:
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Completed:',success,'/',len(images),'Output:',out,flush=True)
if opts['automatically_zip_results'] and success:
    path=root/(out.name+'.zip')
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in out.iterdir():
            if f.is_file():z.write(f,arcname=f.name)
    print('ZIP:',path,flush=True)
if success==0: raise RuntimeError('All images failed; see errors above.')
Path('/content/birefnet_latest_output.txt').write_text(str(out))
"""
script_path=Path('/content/birefnet_run.py'); script_path.write_text(script)
subprocess.run([str(PY),'-u',str(script_path)],check=True)


## 9 · Find latest outputs
Use this cell to review the most recent output folder. Your results are also in Drive even if the runtime disconnects.

In [ ]:
# @title 9 • Results
show_last_20_files = True # @param {type:'boolean'}
from pathlib import Path
import json
cfg=json.loads(Path('/content/birefnet_config.json').read_text())
root=Path(cfg['drive_root'])/'outputs'
folders=sorted([d for d in root.iterdir() if d.is_dir()],key=lambda d:d.stat().st_mtime,reverse=True)
if folders:
    last=folders[0]
    print('Latest results:',last)
    if show_last_20_files:
        for p in sorted(last.iterdir())[:20]: print(' •',p.name)
    print('ZIP:',root/(last.name+'.zip'))
else: print('No processed output yet.')


### Daily use and limitations

- Start a new session: mount Drive, apply storage preferences, verify GPU, recreate UV environment, pick your saved model, and process Drive inputs. Weights are cached, packages are not permanently installed on Colab VMs.
- **Colab Forms** (`# @param`, `# @title`, `# @markdown`) are rendered in the Colab browser. VS Code notebook extensions generally show the code instead of native Colab Forms.
- No `pip` executable is called. `uv pip` is UV's installer. No ordinary `%%bash` commands are needed in the interface.
- This notebook performs **pretrained inference**, not training, fine-tuning, or video segmentation. Quality and performance depend on images, model, GPU driver, and runtime; no model is perfect for every image.
- Sources: [Your BiRefNet repository](https://github.com/orioninsist/BiRefNet), [official BiRefNet HF models](https://huggingface.co/ZhengPeng7), [Colab notebooks](https://colab.research.google.com/).